[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ksankaran/hello-model/blob/main/neural_network.ipynb)

# From Line to Network

In [Hello, Model!](https://medium.com/@v31u/hello-model-build-your-first-ai-model-from-scratch-552ecdffdfe9), we built a model with two knobs: `y = mx + b`. It learned to predict house prices from square footage. The training loop was simple: predict, measure error, adjust knobs, repeat.

But there's a problem. Our model is a straight line. What happens when the real world isn't straight?

In this notebook, we'll:
1. See exactly where a line fails
2. Fix it by adding a single bend (that's a neuron)
3. Stack a few bends together (that's a neural network)
4. Train it with the same loop from last time

Same approach: no frameworks, no libraries, just Python.

---
## Part 1: Where a Line Fails

Let's expand our housing data. This time we have 10 houses, including some larger luxury homes:

In [ ]:
# 10 houses - notice how larger homes get disproportionately expensive
sqft  = [400,  600,  800, 1000, 1200, 1500, 1800, 2200, 2800, 3500]
price = [ 80,  150,  195,  250,  290,  350,  430,  550,  720,  980]

print("Sqft  | Price")
print("------|------")
for s, p in zip(sqft, price):
    print(f"{s:5d} | ${p}k")

print("\nNotice: from 400->1200 sqft, price goes up ~$52/sqft")
print("But from 2200->3500 sqft, price goes up ~$331/sqft")
print("The relationship CURVES upward. A line can't capture this.")

### Let's prove it - train our line model from last time

In [ ]:
# --- Same code from Hello, Model! ---

def line_predict(x, m, b):
    return m * x + b

def line_loss(m, b):
    total = 0
    for x, actual in zip(sqft, price):
        error = line_predict(x, m, b) - actual
        total += error ** 2
    return total / len(sqft)

def line_gradients(m, b):
    gm, gb = 0, 0
    n = len(sqft)
    for x, actual in zip(sqft, price):
        error = line_predict(x, m, b) - actual
        gm += (2/n) * error * x
        gb += (2/n) * error
    return gm, gb

# Train the line
m, b = 0.1, 10.0
for epoch in range(1000):
    gm, gb = line_gradients(m, b)
    m -= 0.0000002 * gm
    b -= 0.0000002 * gb

print(f"Trained line: price = {m:.3f} * sqft + {b:.1f}")
print(f"Loss: {line_loss(m, b):,.1f}")
print()
print("Sqft  | Actual | Line says | Off by")
print("------|--------|-----------|-------")
for x, actual in zip(sqft, price):
    pred = line_predict(x, m, b)
    print(f"{x:5d} | ${actual:5d}k | ${pred:8.1f}k | ${abs(actual-pred):5.1f}k")

The line overshoots small houses and undershoots large ones. It's trying to split the difference, but the data curves and a straight line can't curve.

Let's see this visually:

In [ ]:
def ascii_compare(sqft, price, predict_fn, label="model"):
    """ASCII plot showing data points vs model predictions."""
    width, height = 55, 18
    min_x, max_x = 200, 3800
    min_y, max_y = 30, 1050
    grid = [[' ']*width for _ in range(height)]

    def to_grid(x, y):
        c = int((x - min_x) / (max_x - min_x) * (width - 1))
        r = int((1 - (y - min_y) / (max_y - min_y)) * (height - 1))
        return max(0, min(height-1, r)), max(0, min(width-1, c))

    # Draw predictions
    for px in range(width):
        x = min_x + px / (width-1) * (max_x - min_x)
        y = predict_fn(x)
        if min_y <= y <= max_y:
            r, c = to_grid(x, y)
            grid[r][c] = '-'

    # Draw actual data
    for x, y in zip(sqft, price):
        r, c = to_grid(x, y)
        grid[r][c] = '*'

    for row in grid:
        print(f"  |{''.join(row)}|")
    print(f"  |{'_'*width}|")
    print(f"   {min_x} sqft{' '*(width-16)}{max_x} sqft")
    print(f"   * = actual    - = {label}")

print("Line model vs actual data:\n")
ascii_compare(sqft, price, lambda x: line_predict(x, m, b), "line")

See the gap? The data curves up but our line stays straight. No matter how we adjust `m` and `b`, a line is always straight.

We need a model that can **bend**.

---
## Part 2: A Neuron - A Line That Can Bend

Here's the key insight of neural networks:

> **A neuron is just `y = mx + b` with one extra step: a bend.**

That bend is called an **activation function**. The simplest one is **ReLU** - short for **Rectified Linear Unit**. Fancy name, dead simple idea:

```
ReLU(x) = max(0, x)
```

If the value is negative, output 0. If positive, pass it through.

Let's see what it does:

In [ ]:
def relu(x):
    """ReLU activation - the simplest bend."""
    return max(0.0, x)

# What does ReLU look like?
print("Input | ReLU output")
print("------|------------")
for val in [-3, -2, -1, 0, 1, 2, 3]:
    print(f"  {val:2d}   |   {relu(val):.0f}")

print("\nNegative inputs become 0. Positive inputs pass through unchanged.")
print("That's the 'bend' - it creates a kink at zero.")

### A single neuron

A neuron computes:

```
output = ReLU(w * input + bias)
```

Compare this to our line: `output = m * input + b`

The only difference is the ReLU wrapper. But that one change means the output is no longer a straight line - it's a line that bends at a certain point.

In [ ]:
def neuron(x, w, bias):
    """A single neuron: line + bend."""
    return relu(w * x + bias)

# Let's see different neurons with different weights and biases
print("Three neurons with different knobs, applied to sqft values:\n")

# Each neuron 'activates' (turns on) at a different sqft threshold
configs = [
    (0.001, -0.5, "Activates around 500 sqft"),
    (0.001, -1.5, "Activates around 1500 sqft"),
    (0.001, -2.5, "Activates around 2500 sqft"),
]

test_points = [400, 800, 1200, 1600, 2000, 2400, 2800, 3200]
print(f"{'Sqft':>6} | Neuron 1 | Neuron 2 | Neuron 3")
print(f"-------|----------|----------|----------")
for x in test_points:
    vals = [neuron(x, w, b) for w, b, _ in configs]
    print(f"{x:6d} | {vals[0]:8.2f} | {vals[1]:8.2f} | {vals[2]:8.2f}")

print()
for i, (_, _, desc) in enumerate(configs, 1):
    print(f"Neuron {i}: {desc}")

print("\nEach neuron is 'off' (outputs 0) below its threshold,")
print("then 'turns on' and increases linearly above it.")
print("That's the bend!")

### The key insight

Each neuron contributes a bend at a different point. If we **add them together** with different weights, we get a curve!

Think of it like this:
- Neuron 1 turns on for all houses (base price ramp)
- Neuron 2 turns on for medium+ houses (adds a steeper ramp)
- Neuron 3 turns on only for luxury houses (adds the luxury premium)

Added together: a curve that starts gentle and gets steeper.

Let's see it:

In [ ]:
# Manually picked weights to show the concept
# (Later we'll let gradient descent find these automatically)

def manual_network(x):
    """Three neurons combined - a tiny network."""
    # Hidden layer: three neurons, each with their own weight and bias
    h1 = relu(0.0015 * x - 0.4)   # turns on early, gentle slope
    h2 = relu(0.001 * x - 1.2)    # turns on mid-range
    h3 = relu(0.0008 * x - 2.0)   # turns on for luxury homes

    # Output layer: combine the neurons with weights, add a base
    output = 180 * h1 + 250 * h2 + 400 * h3 + 40
    return output

print("Sqft  | Actual | 3-Neuron Network | Off by")
print("------|--------|------------------|-------")
for x, actual in zip(sqft, price):
    pred = manual_network(x)
    print(f"{x:5d} | ${actual:5d}k | ${pred:15.1f}k | ${abs(actual-pred):5.1f}k")

print("\nNot perfect (we hand-tuned the knobs), but it follows the curve!")

In [ ]:
print("3-neuron network vs actual data:\n")
ascii_compare(sqft, price, manual_network, "network")

The curve follows the data much better than the straight line did. And we built it from the same ingredient - `y = mx + b` - just with bends and stacking.

But we hand-picked those weights. That's cheating. Let's make the computer learn them.

---
## Part 3: The Architecture - What Are We Building?

Before we train, let's be precise about what our neural network looks like:

```
INPUT (sqft)
    |
    v
[Neuron 1]  [Neuron 2]  [Neuron 3]    <- Hidden Layer (3 neurons)
    \           |           /
     \          |          /
      v         v         v
      [    Output Neuron    ]          <- Output Layer (1 neuron)
              |
              v
        PREDICTION (price)
```

**How many knobs (parameters)?**

- Hidden layer: 3 neurons, each with 1 weight + 1 bias = **6 parameters**
- Output layer: 1 neuron with 3 weights (one per hidden neuron) + 1 bias = **4 parameters**
- **Total: 10 parameters** (vs 2 for our line!)

Let's code it up.

In [ ]:
import random

# We need to normalize our data - raw sqft values (400-3500) make
# gradients explode. Dividing by 1000 keeps numbers manageable.
# This is standard practice in ML - always normalize your inputs.

sqft_norm = [s / 1000.0 for s in sqft]    # 0.4 to 3.5
price_norm = [p / 100.0 for p in price]   # 0.8 to 9.8

print("Normalized data (what the network actually sees):")
print("  sqft_norm:", [f"{s:.1f}" for s in sqft_norm])
print("  price_norm:", [f"{p:.1f}" for p in price_norm])
print("\nWe'll convert back to real values for display.")

In [ ]:
class NeuralNetwork:
    """A neural network with 1 hidden layer of 3 neurons.
    
    Architecture: 1 input -> 3 hidden (ReLU) -> 1 output
    Total parameters: 10
    """
    
    def __init__(self):
        # Initialize with small random values
        random.seed(42)  # for reproducibility
        
        # Hidden layer: 3 neurons, each with 1 weight + 1 bias
        self.hw = [random.uniform(-1, 1) for _ in range(3)]  # hidden weights
        self.hb = [random.uniform(-1, 1) for _ in range(3)]  # hidden biases
        
        # Output layer: 3 weights (one per hidden neuron) + 1 bias
        self.ow = [random.uniform(-1, 1) for _ in range(3)]  # output weights
        self.ob = random.uniform(-1, 1)                       # output bias
    
    def forward(self, x):
        """Forward pass - push input through the network."""
        # Step 1: Each hidden neuron computes ReLU(weight * x + bias)
        self.hidden = [relu(self.hw[i] * x + self.hb[i]) for i in range(3)]
        
        # Step 2: Output neuron combines hidden outputs
        output = sum(self.ow[i] * self.hidden[i] for i in range(3)) + self.ob
        
        return output
    
    def get_params(self):
        """Return all 10 parameters as a flat list."""
        return self.hw + self.hb + self.ow + [self.ob]
    
    def set_params(self, params):
        """Set all 10 parameters from a flat list."""
        self.hw = params[0:3]
        self.hb = params[3:6]
        self.ow = params[6:9]
        self.ob = params[9]
    
    def count_params(self):
        return len(self.get_params())

# Create our network
net = NeuralNetwork()
print(f"Network created with {net.count_params()} parameters")
print(f"\nInitial random parameters:")
print(f"  Hidden weights: [{', '.join(f'{w:.3f}' for w in net.hw)}]")
print(f"  Hidden biases:  [{', '.join(f'{b:.3f}' for b in net.hb)}]")
print(f"  Output weights: [{', '.join(f'{w:.3f}' for w in net.ow)}]")
print(f"  Output bias:    {net.ob:.3f}")

### Let's trace through a single prediction

Understanding the forward pass is crucial. Let's watch data flow through the network for one house:

In [ ]:
# Trace a prediction for a 1500 sqft house
x = 1.5  # 1500 sqft, normalized
actual = 3.5  # $350k, normalized

print(f"Input: {x} (1500 sqft, normalized)\n")

print("--- Hidden Layer ---")
for i in range(3):
    raw = net.hw[i] * x + net.hb[i]
    activated = relu(raw)
    print(f"  Neuron {i+1}: {net.hw[i]:.3f} * {x} + {net.hb[i]:.3f} = {raw:.3f} -> ReLU -> {activated:.3f}")

print("\n--- Output Layer ---")
prediction = net.forward(x)
terms = [f"{net.ow[i]:.3f} * {net.hidden[i]:.3f}" for i in range(3)]
print(f"  {' + '.join(terms)} + {net.ob:.3f}")
print(f"  = {prediction:.3f}")

print(f"\nPrediction: ${prediction * 100:.0f}k")
print(f"Actual:     ${actual * 100:.0f}k")
print(f"\nWith random parameters, the prediction is way off. Time to train!")

---
## Part 4: Training - Same Loop, More Knobs

The training loop is **exactly the same** as our line model:

1. **Predict** - push input through the network (forward pass)
2. **Measure error** - compute loss (MSE)
3. **Find the direction** - compute gradients (which way to nudge each knob)
4. **Adjust** - nudge all 10 knobs a tiny bit
5. **Repeat**

The only tricky part is step 3: computing gradients for 10 parameters through multiple layers.

### Computing gradients: the simple way

For our line model, we derived gradient formulas by hand. With a network, the math gets messy (that's what **backpropagation** is - chain rule through each layer).

But there's a simpler way to understand it: **numerical gradients**.

The idea: to find out how much a knob affects the loss, **wiggle it slightly** and see what happens.

In [ ]:
def compute_loss(network):
    """MSE loss - same as before."""
    total = 0
    for x, actual in zip(sqft_norm, price_norm):
        pred = network.forward(x)
        total += (pred - actual) ** 2
    return total / len(sqft_norm)


def compute_numerical_gradients(network, epsilon=0.0001):
    """Compute gradient for each parameter by wiggling it.
    
    For each parameter:
      1. Increase it by a tiny amount (epsilon)
      2. Compute the loss
      3. Decrease it by a tiny amount
      4. Compute the loss
      5. Gradient = (loss_high - loss_low) / (2 * epsilon)
    
    This tells us: if I increase this knob, does the loss go up or down?
    """
    params = network.get_params()
    gradients = []
    
    for i in range(len(params)):
        # Save original value
        original = params[i]
        
        # Wiggle up
        params[i] = original + epsilon
        network.set_params(params)
        loss_up = compute_loss(network)
        
        # Wiggle down
        params[i] = original - epsilon
        network.set_params(params)
        loss_down = compute_loss(network)
        
        # Gradient = slope = rise / run
        grad = (loss_up - loss_down) / (2 * epsilon)
        gradients.append(grad)
        
        # Restore original
        params[i] = original
    
    network.set_params(params)
    return gradients

# Let's see it work
loss = compute_loss(net)
grads = compute_numerical_gradients(net)

print(f"Current loss: {loss:.4f}\n")
print("Gradients (which way to nudge each knob):")
names = ['hw0', 'hw1', 'hw2', 'hb0', 'hb1', 'hb2', 'ow0', 'ow1', 'ow2', 'ob']
for name, grad in zip(names, grads):
    direction = 'decrease' if grad > 0 else 'increase'
    print(f"  {name:4s}: {grad:+.4f}  -> {direction} to reduce loss")

### Why numerical gradients?

This "wiggle each knob" approach is slow (we compute the loss twice per parameter per step), but it's:
- Easy to understand
- Works for any network architecture
- Gives the same answer as backpropagation

In production, frameworks like PyTorch use **backpropagation** (the chain rule applied through each layer) because it's much faster. But the result is identical - a gradient for each parameter.

For our 10-parameter network, the wiggle approach works fine.

---
## Part 5: Let's Train!

Same loop as before. Predict, measure, nudge, repeat.

In [ ]:
# --- TRAINING LOOP ---

net = NeuralNetwork()  # fresh start with random parameters
learning_rate = 0.01
epochs = 2000
history = []

for epoch in range(epochs):
    # 1. How wrong are we?
    loss = compute_loss(net)
    history.append(loss)
    
    # 2. Which way should each knob move?
    grads = compute_numerical_gradients(net)
    
    # 3. Nudge all 10 knobs
    params = net.get_params()
    for i in range(len(params)):
        params[i] -= learning_rate * grads[i]
    net.set_params(params)
    
    # Print progress
    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

final_loss = compute_loss(net)
print(f"\n--- Training complete ---")
print(f"Final loss: {final_loss:.4f}")
print(f"Loss reduction: {(1 - final_loss/history[0])*100:.1f}%")

### How did it do?

In [ ]:
print("Sqft  | Actual | Network  | Line     | Off (Net) | Off (Line)")
print("------|--------|----------|----------|-----------|----------")
for x_raw, x_n, actual_n in zip(sqft, sqft_norm, price_norm):
    pred_net = net.forward(x_n) * 100
    pred_line = line_predict(x_raw, m, b)
    actual = actual_n * 100
    print(f"{x_raw:5d} | ${actual:5.0f}k | ${pred_net:5.0f}k   | ${pred_line:5.0f}k   | ${abs(actual-pred_net):6.1f}k  | ${abs(actual-pred_line):6.1f}k")

In [ ]:
print("Neural network vs actual data:\n")
ascii_compare(sqft, price, lambda x: net.forward(x/1000) * 100, "network")

### What did the network learn?

Let's look at what each neuron ended up doing:

In [ ]:
print("Learned parameters:\n")
print("Hidden layer (each neuron = a bent line):")
for i in range(3):
    # The neuron activates where w*x + b = 0, so x = -b/w
    if net.hw[i] != 0:
        bend_at = -net.hb[i] / net.hw[i]
        bend_sqft = bend_at * 1000
        print(f"  Neuron {i+1}: weight={net.hw[i]:+.3f}, bias={net.hb[i]:+.3f}  "
              f"(bends at ~{bend_sqft:.0f} sqft)")
    else:
        print(f"  Neuron {i+1}: weight={net.hw[i]:+.3f}, bias={net.hb[i]:+.3f}")

print(f"\nOutput layer (how to combine the bends):")
for i in range(3):
    print(f"  Neuron {i+1} contribution: weight={net.ow[i]:+.3f}")
print(f"  Base price (bias): {net.ob:+.3f} (${net.ob * 100:+.0f}k)")

print("\nEach neuron learned to activate at a different sqft threshold.")
print("Together, their weighted sum forms the curve that fits our data.")

---
## Part 6: Watch It Learn

Let's see how the loss dropped during training:

In [ ]:
def ascii_loss_curve(history):
    step = max(1, len(history) // 20)
    sampled = history[::step]
    max_loss = max(sampled)
    min_loss = min(sampled)
    width = 40

    print("Loss over training:\n")
    for i, loss in enumerate(sampled):
        epoch = i * step
        bar = int((loss - min_loss) / (max_loss - min_loss + 0.001) * width)
        print(f"  Epoch {epoch:4d} | {'#' * bar} {loss:.4f}")

    print(f"\n  {history[0]:.4f} -> {history[-1]:.4f}")

ascii_loss_curve(history)

---
## Part 7: Predict New Houses

In [ ]:
new_houses = [500, 900, 1300, 2000, 2500, 3000, 4000]

print("Predictions for houses the network has never seen:\n")
print("Sqft  | Network  | Line")
print("------|----------|------")
for s in new_houses:
    net_pred = net.forward(s / 1000) * 100
    line_pred = line_predict(s, m, b)
    print(f"{s:5d} | ${net_pred:5.0f}k   | ${line_pred:5.0f}k")

print("\nNotice how the network predicts higher prices for large houses")
print("while the line just extends at the same rate.")

---
## Part 8: Backpropagation - The Fast Way

Our wiggle method works, but it's slow. For each training step, we compute the loss **22 times** (2 per parameter, 10 parameters, plus the original). For 10 parameters that's fine, but GPT-4 has 1.8 trillion. Wiggling each one individually would take forever.

**Backpropagation** computes all gradients in **one backward pass** through the network, using the chain rule from calculus. Here's the intuition:

**Forward pass** (left to right):
```
input -> hidden neurons -> output -> loss
```

**Backward pass** (right to left):
```
loss -> how much did output contribute?
     -> how much did each hidden neuron contribute?
     -> how much did each weight contribute?
```

At each step, we ask: "if this value changed a little, how much would the loss change?" The chain rule lets us compute this efficiently by reusing work from later layers.

Let's implement it and verify it gives the same answer as our wiggle method:

In [ ]:
def backprop(network, x, actual):
    """Compute gradients for one example using backpropagation.
    
    Works backward from the loss to each parameter.
    """
    # --- Forward pass (save intermediate values) ---
    # Hidden layer
    h_raw = [network.hw[i] * x + network.hb[i] for i in range(3)]
    h_out = [relu(h_raw[i]) for i in range(3)]
    
    # Output
    pred = sum(network.ow[i] * h_out[i] for i in range(3)) + network.ob
    
    # --- Backward pass ---
    # Step 1: How does the loss change with the prediction?
    # loss = (pred - actual)^2, so d_loss/d_pred = 2 * (pred - actual)
    d_loss = 2 * (pred - actual)
    
    # Step 2: Output layer gradients
    # pred = ow[i] * h_out[i] + ob
    d_ow = [d_loss * h_out[i] for i in range(3)]  # d_loss/d_ow[i]
    d_ob = d_loss                                   # d_loss/d_ob
    
    # Step 3: How much did each hidden neuron contribute to the loss?
    d_h_out = [d_loss * network.ow[i] for i in range(3)]
    
    # Step 4: ReLU gradient - if input was negative, gradient is 0
    d_h_raw = [d_h_out[i] * (1 if h_raw[i] > 0 else 0) for i in range(3)]
    
    # Step 5: Hidden layer weight/bias gradients
    d_hw = [d_h_raw[i] * x for i in range(3)]  # d_loss/d_hw[i]
    d_hb = [d_h_raw[i] for i in range(3)]       # d_loss/d_hb[i]
    
    return d_hw + d_hb + d_ow + [d_ob]


def compute_backprop_gradients(network):
    """Average gradients over all training examples."""
    n = len(sqft_norm)
    total_grads = [0.0] * 10
    
    for x, actual in zip(sqft_norm, price_norm):
        grads = backprop(network, x, actual)
        for i in range(10):
            total_grads[i] += grads[i] / n
    
    return total_grads


# Verify: backprop should give same gradients as the wiggle method
test_net = NeuralNetwork()
numerical = compute_numerical_gradients(test_net)
analytical = compute_backprop_gradients(test_net)

print("Comparing gradient methods:\n")
print(f"{'Param':>5} | {'Numerical':>10} | {'Backprop':>10} | {'Match?':>6}")
print(f"------|------------|------------|-------")
for name, num, ana in zip(names, numerical, analytical):
    match = abs(num - ana) < 0.001
    print(f"{name:>5} | {num:+10.5f} | {ana:+10.5f} | {'yes' if match else 'NO'}")

print("\nSame gradients, but backprop computed them in one pass!")

### Train with backprop - much faster

In [ ]:
# Train with backpropagation
net2 = NeuralNetwork()  # fresh network
learning_rate = 0.01
epochs = 2000
history2 = []

for epoch in range(epochs):
    loss = compute_loss(net2)
    history2.append(loss)
    
    # Backprop instead of numerical gradients
    grads = compute_backprop_gradients(net2)
    
    params = net2.get_params()
    for i in range(len(params)):
        params[i] -= learning_rate * grads[i]
    net2.set_params(params)
    
    if epoch % 400 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f}")

print(f"\nFinal loss: {compute_loss(net2):.4f}")
print(f"Same result, computed much faster.")

---
## Part 9: Why This Matters

Let's zoom out and see what we've built:

```
Line model (Blog 1):     y = m*x + b            2 parameters
Neural network (today):  3 neurons + 1 output   10 parameters
GPT-4:                   billions of neurons    ~1.8 trillion parameters
```

But the **training loop never changed**:

```
1. Forward pass  - push data through the model
2. Compute loss  - how wrong are we?
3. Backward pass - which way should each knob move?
4. Update        - nudge each knob a little
5. Repeat
```

Every neural network ever trained - image classifiers, language models, self-driving cars - uses this exact loop. The differences are:

- **More layers** (depth) - GPT-4 has ~120 layers, not 2
- **More neurons per layer** (width) - thousands, not 3
- **Different architectures** - transformers use "attention" instead of simple ReLU stacking
- **More data** - billions of examples, not 10
- **More compute** - GPU clusters running for months

But the core? Predict. Measure. Adjust. Repeat.

You now understand the foundation that every modern AI system is built on.

---
## Recap: Line vs Network

| | Line (Blog 1) | Neural Network (today) |
|---|---|---|
| **Model** | `y = mx + b` | Neurons with ReLU bends |
| **Parameters** | 2 | 10 |
| **Can learn** | Straight lines | Curves and bends |
| **Training** | Gradient descent | Same gradient descent |
| **Gradients** | Formula | Backpropagation |
| **New concept** | - | Activation function (ReLU) |

## Try It Yourself

Things to experiment with:
- Change from 3 hidden neurons to 5 or 8 - does it fit better?
- Try a learning rate of 0.1 or 0.001 - what happens?
- Add more data points with extreme values
- What if you remove the ReLU (set `relu = lambda x: x`)? Does the network reduce to a line?